# SP519 · Prédiction du flux d'électrons de 1 MeV en LEO (NOAA-15)

## Step 1 : préparation des données et analyse statistique
Le notebook suit, dans l'ordre, les 7 points du Step 1 de l'énoncé :

1. Récupérer les données NOAA-15 SEM-2 et les paramètres solaires pour 2020–2024
2. Nettoyer et synchroniser les données (valeurs manquantes, valeurs invalides, outliers, horodatage)
3. Choisir la région des ceintures de radiation (plage de L) et justifier ce choix
4. Moyenner tous les paramètres sur des intervalles de 12 h
5. Visualiser les séries temporelles (flux vs temps, L vs temps, paramètres solaires vs temps)
6. Statistiques descriptives, relations entre le flux et les paramètres solaires, outliers
7. Déterminer le seuil du 95ᵉ percentile du flux, qui définit les « événements »

(La cellule « 0 » ci-dessous contient seulement les imports et les paramètres.)

## 0. Imports et paramètres

In [ ]:
import os
import re
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import h5py      # les fichiers NOAA-15 sont en netCDF-4 = HDF5
import cdflib    # les fichiers OMNI sont en CDF
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.0, 'font.size': 9})

# Dossiers (relatifs au notebook)
PROJ = os.getcwd()
DATA = os.path.join(PROJ, 'data')
FIG = os.path.join(PROJ, 'figures')
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# Paramètres de l'étude
START, END = pd.Timestamp('2020-01-01'), pd.Timestamp('2025-01-01')   # période [START, END[
STEP = '12h'            # résolution temporelle des prédictions
L_LO, L_HI = 4.0, 6.0   # région de la ceinture retenue (justifiée en section 5)
P_EVENT = 95            # percentile définissant un "événement"

def savefig(fig, name):
    # Sauvegarde des figures pour le rapport
    fig.savefig(os.path.join(FIG, name), dpi=150, bbox_inches='tight')

print('Dossier du projet :', PROJ)

## 1. Récupération des données (NOAA-15 SEM-2 + paramètres solaires, 2020–2024)

> **Code de téléchargement : écrit par l'autre membre du binôme** (à insérer ici).
> Il doit produire les deux fichiers suivants dans `data/`. La cellule ci-dessous ne fait que les lire.

* `noaa15_p6_hourly_Lbins.parquet` : le flux `mep_pro_tel90_flux_p6` de NOAA-15, résumé **heure par heure et par
  tranche de L** (L de 1 à 8 par pas de 0,25, et `Lbin = 10` pour la calotte polaire L ≥ 10).
  Colonnes : `time` (début de l'heure, UTC), `Lbin`, `p6_sum` (somme des flux), `n` (nombre de mesures),
  `omni_p1_sum` / `omni_p1_n` (idem pour les protons 25 MeV, utiles au point 2).
* `omni_hourly_2020_2024.parquet` : paramètres solaires et géomagnétiques OMNI, heure par heure (index `time`, UTC) :
  `V_kms`, `Bz_nT`, `F107_sfu`, `N_cm3`, `P_nPa`, `E_mVm`, `B_nT`, `Kp`, `Dst_nT`, `AE_nT`.

In [ ]:
noaa = pd.read_parquet(os.path.join(DATA, 'noaa15_p6_hourly_Lbins.parquet'))
omni = pd.read_parquet(os.path.join(DATA, 'omni_hourly_2020_2024.parquet'))

L_POLAR = 10.0   # code de la tranche "calotte polaire" (L >= 10)

print('NOAA-15 :', noaa.shape, '|', noaa.time.min(), '->', noaa.time.max())
print('   tranches de L :', [float(l) for l in sorted(noaa.Lbin.unique())])
jours = noaa.time.dt.floor('D').unique()
print('   jours avec données :', len(jours), '/', (END - START).days)
print('OMNI    :', omni.shape, '|', omni.index.min(), '->', omni.index.max())
print('   % manquant :', omni.isna().mean().mul(100).round(2).to_dict())
noaa.head()